# 12장 케라스로 만드는 심층 신경망 — Sequential, Dense, compile, fit, 학습 곡선, 옵티마이저
『머신러닝과 딥러닝 기초』 실습 노트북

다음을 해 봅니다.
1. 11장의 XOR 신경망을 케라스로 다시 만들기
2. 이진 분류: 유방암 데이터
3. 학습 곡선 읽기
4. 다중 분류: 손글씨 숫자
5. 회귀: 펭귄 몸무게
6. 경사 하강법, 모멘텀, Adam 비교

위에서부터 차례로 실행하세요. 학습에 시간이 걸리는 셀이 있습니다(XOR 셀은 1분 안팎).

In [ ]:
# 텐서플로가 찍는 안내 메시지를 줄인다(결과에는 영향 없음)
import os, logging
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
import tensorflow as tf
tf.get_logger().setLevel(logging.ERROR)

## 1. XOR을 케라스로
모델 만들기(`Sequential`, `Dense`) → 컴파일(손실과 `optimizer`) → 학습(`fit`) → 예측(`predict`)의 순서입니다. 구조, 손실, 학습률을 11장과 같게 맞췄습니다(이북 표 12.1).

In [ ]:
import numpy as np
import keras
from keras import layers

keras.utils.set_random_seed(0)          # 무작위 처음 값을 매번 같게
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
Y = np.array([0, 1, 1, 0])

model = keras.Sequential([
    keras.Input(shape=(2,)),                   # 입력 2개
    layers.Dense(2, activation="sigmoid"),     # 은닉층: 뉴런 2개
    layers.Dense(1, activation="sigmoid"),     # 출력층: 뉴런 1개
])
model.compile(optimizer=keras.optimizers.SGD(learning_rate=1.0),
              loss="binary_crossentropy")
hist = model.fit(X, Y, epochs=2000, batch_size=4, verbose=0)

L = hist.history["loss"]
for e in (1, 100, 1000, 2000):
    print(f"에포크 {e:4d}: 손실 {L[e - 1]:.4f}")
print("파라미터 수:", model.count_params())
print("예측:", model.predict(X, verbose=0).ravel().round(3))

학습한 가중치를 꺼내 봅니다. 은닉 뉴런이 무엇을 계산하는지 이북 12.1절의 설명과 맞춰 보세요.

In [ ]:
W1, b1, W2, b2 = model.get_weights()
print("은닉층 가중치:\n", W1.round(2))
print("은닉층 편향:", b1.round(2))
print("출력층 가중치:", W2.ravel().round(2), " 편향:", b2.round(2))

## 2. 이진 분류: 유방암 데이터
4장, 9장과 같은 나눔과 표준화입니다. 출력층은 시그모이드 뉴런 하나, 손실은 이진 교차 엔트로피, `validation_split=0.2`로 학습 데이터의 마지막 20%를 검증 데이터로 씁니다.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

cancer = load_breast_cancer(as_frame=True)   # 4장, 9장의 데이터
X = cancer.data.values
y = (cancer.target == 0).astype(int).values  # 악성이면 1
Xtr, Xte, ytr, yte = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)
sc = StandardScaler().fit(Xtr)               # 표준화(3장)
Xtr_s, Xte_s = sc.transform(Xtr), sc.transform(Xte)

keras.utils.set_random_seed(42)
model = keras.Sequential([
    keras.Input(shape=(30,)),                # 특징 30개
    layers.Dense(16, activation="relu"),     # 은닉층: ReLU 16개
    layers.Dense(1, activation="sigmoid"),   # 출력: 악성일 확률
])
model.compile(optimizer="adam", loss="binary_crossentropy",
              metrics=["accuracy"])
hist = model.fit(Xtr_s, ytr, epochs=100, batch_size=32,
                 validation_split=0.2, verbose=0)
print("파라미터 수:", model.count_params())
H = hist.history
for e in (1, 10, 100):
    print(f"에포크 {e:3d}: 학습 손실 {H['loss'][e - 1]:.3f}, "
          f"검증 손실 {H['val_loss'][e - 1]:.3f}, "
          f"검증 정확도 {H['val_accuracy'][e - 1]:.1%}")

테스트 데이터로 평가합니다. 9장의 로지스틱 회귀는 테스트 정확도 96.5%였습니다.

In [ ]:
loss, acc = model.evaluate(Xte_s, yte, verbose=0)
print(f"테스트 손실 {loss:.3f}, 테스트 정확도 {acc:.1%}")
p = model.predict(Xte_s[:3], verbose=0).ravel()
print("처음 세 건의 악성 확률:", p.round(3), " 정답:", yte[:3])

## 3. 학습 곡선
같은 모델을 300 에포크 학습해 학습 손실과 검증 손실을 비교합니다.

In [ ]:
keras.utils.set_random_seed(42)
long = keras.Sequential([
    keras.Input(shape=(30,)),
    layers.Dense(16, activation="relu"),
    layers.Dense(1, activation="sigmoid"),
])
long.compile(optimizer="adam", loss="binary_crossentropy",
             metrics=["accuracy"])
H = long.fit(Xtr_s, ytr, epochs=300, batch_size=32,
             validation_split=0.2, verbose=0).history
best = int(np.argmin(H["val_loss"])) + 1   # 검증 손실 최소 에포크
print(f"검증 손실이 가장 낮은 에포크: {best}")
for e in (best, 300):
    print(f"에포크 {e:3d}: 학습 손실 {H['loss'][e - 1]:.3f}, "
          f"검증 손실 {H['val_loss'][e - 1]:.3f}")
loss, acc = long.evaluate(Xte_s, yte, verbose=0)
print(f"300 에포크 뒤 테스트 손실 {loss:.3f}, 정확도 {acc:.1%}")

학습 곡선을 그려 봅니다(이북 그림 12.1). 점선 세로선이 검증 손실이 가장 낮은 에포크입니다.

In [ ]:
import matplotlib.pyplot as plt

fig, axs = plt.subplots(1, 2, figsize=(10, 3.5))
axs[0].plot(H["loss"], label="train loss"); axs[0].plot(H["val_loss"], "--", label="validation loss")
axs[0].axvline(best - 1, color="gray", ls=":"); axs[0].set_ylim(0, 0.35); axs[0].set_xlabel("epoch"); axs[0].legend()
axs[1].plot(H["accuracy"], label="train accuracy"); axs[1].plot(H["val_accuracy"], "--", label="validation accuracy")
axs[1].set_ylim(0.5, 1.02); axs[1].set_xlabel("epoch"); axs[1].legend()
plt.tight_layout(); plt.show()

## 4. 다중 분류: 손글씨 숫자
출력층은 소프트맥스 뉴런 10개, 손실은 `sparse_categorical_crossentropy`(정답이 숫자 하나일 때)입니다. 10장의 `MLPClassifier`는 테스트 정확도 97.8%였습니다.

In [ ]:
from sklearn.datasets import load_digits

Xd, yd = load_digits(return_X_y=True)        # 6장, 10장의 데이터
Dtr, Dte, dtr, dte = train_test_split(
    Xd, yd, test_size=0.25, random_state=42, stratify=yd)
sd = StandardScaler().fit(Dtr)
Dtr_s, Dte_s = sd.transform(Dtr), sd.transform(Dte)

keras.utils.set_random_seed(42)
dnet = keras.Sequential([
    keras.Input(shape=(64,)),                # 픽셀 64개
    layers.Dense(64, activation="relu"),
    layers.Dense(10, activation="softmax"),  # 숫자 0~9의 확률
])
dnet.compile(optimizer="adam",
             loss="sparse_categorical_crossentropy",
             metrics=["accuracy"])
dnet.fit(Dtr_s, dtr, epochs=50, batch_size=32,
         validation_split=0.2, verbose=0)
loss, acc = dnet.evaluate(Dte_s, dte, verbose=0)
print("파라미터 수:", dnet.count_params())
print(f"테스트 손실 {loss:.3f}, 테스트 정확도 {acc:.1%}")

첫 번째 테스트 이미지의 확률 10개와 손실을 봅니다(이북 그림 12.2).

In [ ]:
p = dnet.predict(Dte_s[:1], verbose=0)[0]   # 첫 테스트 이미지
print("정답:", dte[0])
print("확률:", p.round(3))
print("확률의 합:", round(float(p.sum()), 3))
print("가장 큰 확률의 숫자:", p.argmax())
print("손실:", round(float(-np.log(p[dte[0]])), 3))   # −ln(정답 확률)

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(9, 3))
axs[0].imshow(Dte[0].reshape(8, 8), cmap="gray_r"); axs[0].set_title(f"label {dte[0]}"); axs[0].axis("off")
axs[1].bar(range(10), p); axs[1].set_xticks(range(10)); axs[1].set_xlabel("digit"); axs[1].set_ylabel("probability")
plt.show()

## 5. 회귀: 펭귄 몸무게
출력층에 활성화 함수를 쓰지 않고 손실은 `mse`입니다. 몸무게는 kg으로 바꾸어 학습합니다. 7장의 선형 회귀(특징 3개 + 종)는 RMSE 305.2 g, R² 0.863이었습니다.

In [ ]:
import pandas as pd
import seaborn as sns
from sklearn.metrics import mean_squared_error, r2_score

num = ["bill_length_mm", "bill_depth_mm",
       "flipper_length_mm", "body_mass_g"]
df = sns.load_dataset("penguins").dropna(subset=num)
df = df.reset_index(drop=True)            # 342마리(7장과 같다)
feats = ["flipper_length_mm", "bill_length_mm", "bill_depth_mm"]
tr, te = train_test_split(df, test_size=0.25, random_state=42,
                          stratify=df["species"])

def prep(d):                              # 특징 3개 + 종의 원-핫
    sp = pd.get_dummies(d["species"], dtype=float)
    return pd.concat([d[feats], sp], axis=1).values

Ptr, Pte = prep(tr), prep(te)
mtr = tr["body_mass_g"].values / 1000           # 몸무게를 kg으로
mte = te["body_mass_g"].values / 1000
sp_ = StandardScaler().fit(Ptr)
Ptr_s, Pte_s = sp_.transform(Ptr), sp_.transform(Pte)

keras.utils.set_random_seed(42)
reg = keras.Sequential([
    keras.Input(shape=(6,)),
    layers.Dense(16, activation="relu"),
    layers.Dense(1),                      # 출력층: 활성화 함수 없음
])
reg.compile(optimizer=keras.optimizers.Adam(learning_rate=0.01),
            loss="mse", metrics=["mae"])
reg.fit(Ptr_s, mtr, epochs=200, batch_size=32,
        validation_split=0.2, verbose=0)
pred = reg.predict(Pte_s, verbose=0).ravel()
rmse = mean_squared_error(mte, pred) ** 0.5 * 1000
print(f"RMSE {rmse:.1f} g, R² {r2_score(mte, pred):.3f}")

## 6. 경사 하강법, 모멘텀, Adam
손글씨 숫자 신경망을 세 방법으로 30 에포크씩 학습합니다(이북 그림 12.3 ②).

In [ ]:
from keras.optimizers import SGD, Adam

for name, opt in [("SGD", SGD(learning_rate=0.01)),
                  ("모멘텀", SGD(learning_rate=0.01,
                                momentum=0.9)),
                  ("Adam", Adam(learning_rate=0.001))]:
    keras.utils.set_random_seed(42)         # 처음 가중치를 같게
    m = keras.Sequential([
        keras.Input(shape=(64,)),
        layers.Dense(64, activation="relu"),
        layers.Dense(10, activation="softmax"),
    ])
    m.compile(optimizer=opt,
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])
    H = m.fit(Dtr_s, dtr, epochs=30, batch_size=32,
              validation_split=0.2, verbose=0).history
    v, a = H["val_loss"], H["val_accuracy"]
    print(f"{name}: 검증 손실 에포크 5 {v[4]:.3f}, 30 {v[29]:.3f}, "
          f"검증 정확도 {a[29]:.1%}")

에포크마다의 검증 손실을 그려 봅니다.

In [ ]:
curves = {}
for name, opt in [("SGD", SGD(learning_rate=0.01)),
                  ("momentum", SGD(learning_rate=0.01, momentum=0.9)),
                  ("Adam", Adam(learning_rate=0.001))]:
    keras.utils.set_random_seed(42)
    m = keras.Sequential([keras.Input(shape=(64,)), layers.Dense(64, activation="relu"),
                          layers.Dense(10, activation="softmax")])
    m.compile(optimizer=opt, loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    curves[name] = m.fit(Dtr_s, dtr, epochs=30, batch_size=32, validation_split=0.2, verbose=0).history["val_loss"]
for name, v in curves.items():
    plt.plot(range(1, 31), v, label=name)
plt.xlabel("epoch"); plt.ylabel("validation loss"); plt.legend(); plt.show()

---
## 실습 문제

**문제 1.** 유방암 신경망의 은닉 뉴런 수를 4, 16, 64, 256으로 바꾸어(다른 설정은 2절과 같게, 100 에포크) 파라미터 수와 테스트 정확도를 출력하세요. 뉴런을 늘리면 늘 좋아지나요?

In [ ]:
# 문제 1 풀이

**문제 2.** 손글씨 숫자의 레이블을 `keras.utils.to_categorical(dtr, 10)`으로 원-핫으로 바꾸고, 손실을 `categorical_crossentropy`로 바꾸어 4절과 같은 설정(같은 시드, 50 에포크)으로 학습하세요. 테스트 정확도가 4절과 같은지 확인하세요.

In [ ]:
# 문제 2 풀이

**문제 3.** 펭귄 몸무게 회귀에서 (가) 학습률을 기본값 0.001로 두었을 때, (나) 몸무게를 kg으로 바꾸지 않고 g 그대로 학습했을 때(학습률 0.01)의 테스트 RMSE를 구해 5절의 결과(306.5 g)와 비교하세요. 왜 차이가 날까요?

In [ ]:
# 문제 3 풀이

**문제 4.** 6절의 비교에서 (가) SGD의 학습률을 0.1로 키운 경우, (나) 모멘텀 계수를 0.5와 0.99로 바꾼 경우(학습률 0.01), (다) Adam의 학습률을 0.01로 키운 경우의 30 에포크 뒤 검증 손실과 검증 정확도를 구하세요. 어떤 경향이 보이나요?

In [ ]:
# 문제 4 풀이